### Results

In [4]:
import pandas as pd

files = {
    'Baseline'   : 'results/baseline/ml-100k/results.csv',
    'CTGAN'      : 'results/ctgan/ml-100k/results.csv',
    'GReat'      : 'results/great/ml-100k/results.csv',
    'TAEGAN'     : 'results/taegan/ml-100k/results.csv',
    'TabFairGan' : 'results/tabfairgan/ml-100k/results.csv',
    'CFGAN'      : 'results/cfgan/ml-100k/results.csv',
    'DECAF'      : 'results/decaf/ml-100k/results.csv',
}

metrics = ['HR@10', 'NDCG@10', 'MAE', 'Macro_F1', 'Precision', 'U_abs', 'MAED', 'TI', 'CNT']
dfs = {name: pd.read_csv(path) for name, path in files.items()}

rows = []
for name, df in dfs.items():
    for _, row in df.iterrows():
        entry = {'GAN': name, 'Model': row['Model']}
        for m in metrics:
            entry[m] = row[m]
        rows.append(entry)

combined = pd.DataFrame(rows)
combined = combined.set_index(['GAN', 'Model'])

print("=" * 120)
print("FULL RESULTS TABLE")
print("=" * 120)
print(combined.to_string())

baseline_df = dfs['Baseline'].set_index('Model')
lower_is_better = ['MAE', 'U_abs', 'MAED', 'TI']
higher_is_better = ['HR@10', 'NDCG@10', 'Macro_F1', 'Precision', 'CNT']

print("\n\n" + "=" * 120)
print("IMPROVEMENT vs BASELINE (✓ = improved, ✗ = worse, ~ = no change)")
print("=" * 120)

improvement_rows = []
for name, df in dfs.items():
    if name == 'Baseline':
        continue
    for _, row in df.iterrows():
        model = row['Model']
        entry = {'GAN': name, 'Model': model}
        for m in metrics:
            baseline_val = baseline_df.loc[model, m]
            current_val = row[m]
            diff = current_val - baseline_val
            if abs(diff) < 1e-6:
                symbol = '~'
            elif m in lower_is_better:
                symbol = '✓' if diff < 0 else '✗'
            else:
                symbol = '✓' if diff > 0 else '✗'
            entry[m] = f"{symbol} ({diff:+.4f})"
        improvement_rows.append(entry)

improvement_df = pd.DataFrame(improvement_rows).set_index(['GAN', 'Model'])
print(improvement_df.to_string())

print("\n\n" + "=" * 120)
print("AVERAGED IMPROVEMENT vs BASELINE (per GAN method, averaged across all model scales)")
print("=" * 120)

baseline_avg = dfs['Baseline'][metrics].mean()

avg_rows = []
for name, df in dfs.items():
    if name == 'Baseline':
        continue
    gan_avg = df[metrics].mean()
    entry = {'GAN': name}
    for m in metrics:
        diff = gan_avg[m] - baseline_avg[m]
        if abs(diff) < 1e-6:
            symbol = '~'
        elif m in lower_is_better:
            symbol = '✓' if diff < 0 else '✗'
        else:
            symbol = '✓' if diff > 0 else '✗'
        entry[m] = f"{symbol} ({diff:+.4f})"
    avg_rows.append(entry)

avg_df = pd.DataFrame(avg_rows).set_index('GAN')
print(avg_df.to_string())

# ─── SHARED HELPERS ───────────────────────────────────────────────────────────

model_order = ['tiny', 'small', 'medium', 'large', 'xlarge']

def compute_best(metrics_subset):
    best = {m: {} for m in metrics_subset}
    for m in metrics_subset:
        for scale in model_order:
            vals = [dfs[gan].set_index('Model').loc[scale, m] for gan in dfs]
            best[m][scale] = min(vals) if m in lower_is_better else max(vals)
    return best

def fmt(val, metric, scale, best):
    s = f"{val:.3f}"
    if abs(val - best[metric][scale]) < 1e-6:
        s = r"\textbf{" + s + "}"
    return s

fairness_metrics = ['CNT', 'TI', 'MAED', 'U_abs']
rec_metrics      = ['HR@10', 'NDCG@10', 'MAE', 'Macro_F1', 'Precision']
best_f = compute_best(fairness_metrics)
best_r = compute_best(rec_metrics)

# ─── CONFERENCE STYLE ─────────────────────────────────────────────────────────
# l|l|c... columns, no outer pipes, \hline between groups, blank name cell

def print_conference_fairness():
    print(r"""\begin{table}[htbp]
    \centering
    \caption{Fairness Metric Evaluation Results on MovieLens-100k Dataset}
    \label{tab:ml100k_fairness_full}
    \vspace{1mm}
    \resizebox{\linewidth}{!}{%
        \begin{tabular}{l|l|c|c|c|c}
            \hline
            \textbf{Model} & \textbf{Scale} & \textbf{CNT} & \textbf{TI} & \textbf{MAED} & \textbf{$U_{abs}$} \\ \hline""")
    for gan_name, df in dfs.items():
        df_indexed = df.set_index('Model')
        for i, model in enumerate(model_order):
            row  = df_indexed.loc[model]
            cnt  = fmt(row['CNT'],   'CNT',   model, best_f)
            ti   = fmt(row['TI'],    'TI',    model, best_f)
            maed = fmt(row['MAED'],  'MAED',  model, best_f)
            uabs = fmt(row['U_abs'], 'U_abs', model, best_f)
            name_cell = gan_name if i == 0 else ''
            print(f"            {name_cell} & {model} & {cnt} & {ti} & {maed} & {uabs} \\\\")
        print(r"            \hline")
    print(r"""        \end{tabular}%
    }
\end{table}""")

def print_conference_recommendation():
    print(r"""\begin{table}[htbp]
    \centering
    \caption{Recommendation Metric Evaluation Results on MovieLens-100k Dataset}
    \label{tab:ml100k_recommendation_full}
    \vspace{1mm}
    \resizebox{\linewidth}{!}{%
        \begin{tabular}{l|l|c|c|c|c|c}
            \hline
            \textbf{Model} & \textbf{Scale} & \textbf{HR@10} & \textbf{NDCG@10} & \textbf{MAE} & \textbf{F1} & \textbf{Prec} \\ \hline""")
    for gan_name, df in dfs.items():
        df_indexed = df.set_index('Model')
        for i, model in enumerate(model_order):
            row  = df_indexed.loc[model]
            hr   = fmt(row['HR@10'],     'HR@10',     model, best_r)
            ndcg = fmt(row['NDCG@10'],   'NDCG@10',   model, best_r)
            mae  = fmt(row['MAE'],       'MAE',        model, best_r)
            f1   = fmt(row['Macro_F1'],  'Macro_F1',  model, best_r)
            prec = fmt(row['Precision'], 'Precision', model, best_r)
            name_cell = gan_name if i == 0 else ''
            print(f"            {name_cell} & {model} & {hr} & {ndcg} & {mae} & {f1} & {prec} \\\\")
        print(r"            \hline")
    print(r"""        \end{tabular}%
    }
\end{table}""")

# ─── THESIS STYLE ─────────────────────────────────────────────────────────────
# |l|c|c... outer pipes, \multirow, \cline between scale rows, \hline between groups

def print_thesis_fairness():
    print(r"""\begin{table}[htbp]
    \centering
    \caption{Hasil Evaluasi Metrik Keadilan pada Dataset MovieLens-100k}
    \label{tab:ml100k_fairness_full}
    \vspace{3mm}
    \setlength{\tabcolsep}{11pt}
    \resizebox{\linewidth}{!}{%
        \begin{tabular}{|l|c|c|c|c|c|}
            \hline
            \textbf{Model} & \textbf{Scale} & \textbf{CNT} & \textbf{TI} & \textbf{MAED} & \textbf{$U_{abs}$} \\ \hline""")
    for gan_name, df in dfs.items():
        df_indexed = df.set_index('Model')
        print(f"            \\multirow{{5}}{{*}}{{{gan_name}}}", end="")
        for i, model in enumerate(model_order):
            row  = df_indexed.loc[model]
            cnt  = fmt(row['CNT'],   'CNT',   model, best_f)
            ti   = fmt(row['TI'],    'TI',    model, best_f)
            maed = fmt(row['MAED'],  'MAED',  model, best_f)
            uabs = fmt(row['U_abs'], 'U_abs', model, best_f)
            is_last = (i == len(model_order) - 1)
            sep = r" \\ \hline" if is_last else r" \\ \cline{2-6}"
            if i == 0:
                print(f" & {model} & {cnt} & {ti} & {maed} & {uabs}{sep}")
            else:
                print(f"             & {model} & {cnt} & {ti} & {maed} & {uabs}{sep}")
    print(r"""        \end{tabular}%
    }
\end{table}""")

def print_thesis_recommendation():
    print(r"""\begin{table}[htbp]
    \centering
    \caption{Hasil Evaluasi Metrik Rekomendasi pada Dataset MovieLens-100k}
    \label{tab:ml100k_recommendation_full}
    \vspace{3mm}
    \setlength{\tabcolsep}{11pt}
    \resizebox{\linewidth}{!}{%
        \begin{tabular}{|l|c|c|c|c|c|c|}
            \hline
            \textbf{Model} & \textbf{Scale} & \textbf{HR@10} & \textbf{NDCG@10} & \textbf{MAE} & \textbf{F1} & \textbf{Prec} \\ \hline""")
    for gan_name, df in dfs.items():
        df_indexed = df.set_index('Model')
        print(f"            \\multirow{{5}}{{*}}{{{gan_name}}}", end="")
        for i, model in enumerate(model_order):
            row  = df_indexed.loc[model]
            hr   = fmt(row['HR@10'],     'HR@10',     model, best_r)
            ndcg = fmt(row['NDCG@10'],   'NDCG@10',   model, best_r)
            mae  = fmt(row['MAE'],       'MAE',        model, best_r)
            f1   = fmt(row['Macro_F1'],  'Macro_F1',  model, best_r)
            prec = fmt(row['Precision'], 'Precision', model, best_r)
            is_last = (i == len(model_order) - 1)
            sep = r" \\ \hline" if is_last else r" \\ \cline{2-7}"
            if i == 0:
                print(f" & {model} & {hr} & {ndcg} & {mae} & {f1} & {prec}{sep}")
            else:
                print(f"             & {model} & {hr} & {ndcg} & {mae} & {f1} & {prec}{sep}")
    print(r"""        \end{tabular}%
    }
\end{table}""")

# ─── OUTPUT ───────────────────────────────────────────────────────────────────

print("\n\n" + "=" * 120)
print("LATEX CONFERENCE STYLE — FAIRNESS")
print("=" * 120)
print_conference_fairness()

print("\n\n" + "=" * 120)
print("LATEX CONFERENCE STYLE — RECOMMENDATION")
print("=" * 120)
print_conference_recommendation()

print("\n\n" + "=" * 120)
print("LATEX THESIS STYLE — FAIRNESS")
print("=" * 120)
print_thesis_fairness()

print("\n\n" + "=" * 120)
print("LATEX THESIS STYLE — RECOMMENDATION")
print("=" * 120)
print_thesis_recommendation()

FULL RESULTS TABLE
                      HR@10   NDCG@10       MAE  Macro_F1  Precision     U_abs      MAED        TI       CNT
GAN        Model                                                                                            
Baseline   large   0.977636  0.809880  0.370068  0.701533   0.706087  0.003633  0.003906  0.133384  0.633183
           medium  0.977636  0.809151  0.368004  0.698310   0.708433  0.010130  0.005191  0.129474  0.638101
           small   0.976571  0.806860  0.368775  0.701773   0.708422  0.009825  0.004886  0.135119  0.630819
           tiny    0.977636  0.806353  0.375585  0.699740   0.704443  0.002140  0.002799  0.114581  0.625070
           xlarge  0.977636  0.809068  0.373012  0.700648   0.706303  0.008871  0.006604  0.128420  0.632782
CTGAN      large   0.977636  0.809781  0.398317  0.698907   0.704989  0.004050  0.003485  0.077940  0.629157
           medium  0.977636  0.809539  0.396544  0.699273   0.708276  0.005472  0.005909  0.079768  0.632542


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
os.makedirs('figures', exist_ok=True)
files = {
    'Baseline': 'results/baseline/ml-100k/results.csv',
    'CTGAN': 'results/ctgan/ml-100k/results.csv',
    'GReat': 'results/great/ml-100k/results.csv',
    'TAEGAN': 'results/taegan/ml-100k/results.csv',
    'TabFairGan': 'results/tabfairgan/ml-100k/results.csv',
    'CFGAN': 'results/cfgan/ml-100k/results.csv',
    'DECAF': 'results/decaf/ml-100k/results.csv',
}
dfs = {name: pd.read_csv(path) for name, path in files.items()}
fairness_metrics = ['CNT', 'TI', 'MAED', 'U_abs']
recommendation_metrics = ['HR@10', 'NDCG@10', 'MAE', 'Macro_F1', 'Precision']
colors = ['#8DD3C7', '#FFFFB3', '#BEBADA', '#FB8072', '#80B1D3', '#FDB462', '#B3DE69']
print("Creating bar charts...")
for metric in fairness_metrics + recommendation_metrics:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    gan_methods = list(dfs.keys())
    averages = [dfs[name][metric].mean() for name in gan_methods]
    
    x_pos = np.arange(len(gan_methods))
    
    bars = ax.bar(x_pos, averages, color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
    
    for i, (bar, avg) in enumerate(zip(bars, averages)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{avg:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_xlabel('GAN Method', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{metric}', fontsize=13, fontweight='bold')
    ax.set_title(f'Average {metric} Across Different GAN Methods', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(gan_methods, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if metric in ['CNT', 'HR@10', 'NDCG@10', 'Macro_F1', 'Precision']:
        ax.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig(f'figures/barchart_{metric.replace("@", "_at_")}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: barchart_{metric.replace('@', '_at_')}.png")
print("\n✓ All bar charts created successfully!")

Creating bar charts...
✓ Saved: barchart_CNT.png
✓ Saved: barchart_TI.png
✓ Saved: barchart_MAED.png
✓ Saved: barchart_U_abs.png
✓ Saved: barchart_HR_at_10.png
✓ Saved: barchart_NDCG_at_10.png
✓ Saved: barchart_MAE.png
✓ Saved: barchart_Macro_F1.png
✓ Saved: barchart_Precision.png

✓ All bar charts created successfully!
